<a href="https://colab.research.google.com/github/saifmukadam10/policy-Improvment---RL/blob/main/Policy_Improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [3]:
transitions = {
    # State 1 transitions
    (1,2,2): 0.8,(1,2,1):0.1,(1,2,-1):0.1, (1,4,-1): 0.8,(1,4,2):0.1,(1,4,-2):0.1,
    # State 2 transitions
    (2,3,2): 0.8,(2,3,-1):0.1,(2,3,1):0.1, (2,1,-2): 0.8,(2,1,1):0.1,(2,1,-1):0.1,
    # State 3 transitions (kept commented as they are unclear/incomplete)
    # (3,2,-2): 0.8,(3,2,1):0.1,(3,2,-1):0.1, (3,6,-1): 0.8,(3,6,2):0.1,(3,6,-2):0.1,
    # (3,2,-2): 0,(3,2,1):0,(3,2,-1):0, (3,6,-1): 0,(3,6,2):0,(3,6,-2):0,
    # State 4 transitions
    (4,1,1): 0.8,(4,1,-2):0.1,(4,1,2):0.1, (4,5,2): 0.8,(4,5,1):0.1,(4,5,-1):0.1,
    # State 5 transitions
    (5,2,1): 0.8,(5,2,-2):0.1,(5,2,2):0.1, (5,4,-2): 0.8,(5,4,1):0.1,(5,4,-1):0.1,
    # State 6 transitions
    (6,3,1): 0.8,(6,3,-2):0.1,(6,3,2):0.1, (6,5,-2): 0.8,(6,5,1):0.1,(6,5,-1):0.1,
    # State 7 transitions
    (7,4,1): 0.8,(7,4,2):0.1,(7,4,-2):0.1, (7,8,2): 0.8,(7,8,1):0.1,(7,8,-1):0.1,
    # State 8 transitions
    (8,9,2): 0.8,(8,9,-1):0.1,(8,9,1):0.1, (8,7,-2): 0.8,(8,7,1):0.1,(8,7,-1):0.1,
    # State 9 transitions
    (9,6,1): 0.8,(9,6,2):0.1,(9,6,-2):0.1, (9,8,-2): 0.8,(9,8,1):0.1,(9,8,-1):0.1
}

In [4]:
transitions

{(1, 2, 2): 0.8,
 (1, 2, 1): 0.1,
 (1, 2, -1): 0.1,
 (1, 4, -1): 0.8,
 (1, 4, 2): 0.1,
 (1, 4, -2): 0.1,
 (2, 3, 2): 0.8,
 (2, 3, -1): 0.1,
 (2, 3, 1): 0.1,
 (2, 1, -2): 0.8,
 (2, 1, 1): 0.1,
 (2, 1, -1): 0.1,
 (4, 1, 1): 0.8,
 (4, 1, -2): 0.1,
 (4, 1, 2): 0.1,
 (4, 5, 2): 0.8,
 (4, 5, 1): 0.1,
 (4, 5, -1): 0.1,
 (5, 2, 1): 0.8,
 (5, 2, -2): 0.1,
 (5, 2, 2): 0.1,
 (5, 4, -2): 0.8,
 (5, 4, 1): 0.1,
 (5, 4, -1): 0.1,
 (6, 3, 1): 0.8,
 (6, 3, -2): 0.1,
 (6, 3, 2): 0.1,
 (6, 5, -2): 0.8,
 (6, 5, 1): 0.1,
 (6, 5, -1): 0.1,
 (7, 4, 1): 0.8,
 (7, 4, 2): 0.1,
 (7, 4, -2): 0.1,
 (7, 8, 2): 0.8,
 (7, 8, 1): 0.1,
 (7, 8, -1): 0.1,
 (8, 9, 2): 0.8,
 (8, 9, -1): 0.1,
 (8, 9, 1): 0.1,
 (8, 7, -2): 0.8,
 (8, 7, 1): 0.1,
 (8, 7, -1): 0.1,
 (9, 6, 1): 0.8,
 (9, 6, 2): 0.1,
 (9, 6, -2): 0.1,
 (9, 8, -2): 0.8,
 (9, 8, 1): 0.1,
 (9, 8, -1): 0.1}

In [7]:
#Q-Values = expected future utility from a q-state (chance node) In [4]:
class MDP:
    def __init__(self, states, action, transition, start,policy, discountf=.99):
        self.states = states
        self.action = action
        self.transition = transition
        self.start = start
        self.discountf = discountf
        self.policy = policy
    def R(self, s):
      #return reward for this state.
      return self.states[s]
    def T(self, s, s_dash,a):
      #for a state and an action, return probability of s'.
      return self.transition[(s,s_dash,a)]
def getpath(newp,start):
    currentstate = start
    itr = 0
    while ((newp[currentstate-1] != 0) and (itr <=9)):
        itr+=1
        x = newp[currentstate-1]
        if (x == 2):
            print("Right")
            if (currentstate % 3 != 0):
                currentstate+=1
        elif (x == -2):
            print("Left")
            if (currentstate % 3 != 1):
                currentstate-=1
        elif (x == -1):
            print("Down")
            if (currentstate <= 6):
                currentstate+=3
        elif (x == 1):
            print("Up")
            if (currentstate >= 4):
                currentstate-=3
    return

In [8]:
def ValueIteration(numofIteration, mdp):
    rewards = mdp.states
    actions = mdp.action

    vPrevois = [[0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0]]
    vCurrent = [[0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0],
                [0,0], [0,0], [0,0]]

    for i in range(numofIteration):
        print("itration ", i)
        for s in range(9):
            delta = 0
            Q = [0, 0, 0, 0]
            for a in range(4):
                for s2 in range(9):
                    if (s+1,s2+1,actions[a]) in mdp.transition:
                        #print("s s' a",s+1,s2+1,actions[a], " T ", mdp.transition[(s+1,s2+1
                        Q[a] = Q[a] + mdp.transition[(s+1,s2+1,actions[a])] * (rewards[s]) # Assuming rewards[s] should be used here and not rewards[s2]

            max_index = Q.index(max(Q))
            #print("Q ", Q, " max Q ", max(Q)," max_index ",max_index)
            #print("node ", s+1)
            if s == 0 or s == 2:
                vCurrent[s] = [max(Q),0]
            else:
                vCurrent[s] = [max(Q),actions[max_index]]

            delta = max(delta, abs(vCurrent[s][0] - vPrevois[s][0]))
            vPrevois[s] = vCurrent[s]

        #check for convergence, if values converged then return V
        # Correcting the calculation of the convergence threshold and variable name
        convergence_threshold = (1 - mdp.discountf) / (2 * mdp.discountf)
        print("convergence ", convergence_threshold, " delta ", delta)
        if delta < convergence_threshold and delta != 0:
            print("Early Stop at itration ", i)
            return vCurrent

    return vCurrent

In [9]:
import numpy as np

def PolicyIteration(n_iter, mdp):
    Vprev = [0.0] * 9  # Initialize value function (0-indexed)
    newPolicy = list(mdp.policy)  # Make a mutable copy of the initial policy

    # The following lines appear to be comments or output from a previous run and are commented out.
    # Early Stop at itration 3
    # policy for r = 100
    # [ 0 -2 0 1 -2 -2 1 1 -2]
    # path from start point
    # Up
    # Left
    # Up
    # Early Stop at itration 3
    # policy for r = 3

    for itr in range(n_iter):
        policy_changed_count = 0  # Flag to detect if policy changed in this iteration

        # --- Policy Evaluation Step (single pass Bellman expectation update) ---
        V_evaluated_for_current_policy = [0.0] * 9  # Values calculated for the current policy

        for s_idx in range(9):  # Iterate through states (0-indexed for arrays)
            s = s_idx + 1  # Convert to 1-indexed for MDP functions like T and transition keys
            current_action = newPolicy[s_idx]
            expected_value_for_s = 0.0

            # Sum over all possible next states s_prime
            for s_prime_idx in range(9):  # Iterate through all potential next states (0-indexed)
                s_prime = s_prime_idx + 1  # Convert to 1-indexed
                if (s, s_prime, current_action) in mdp.transition: # Check if this transition exists
                    prob = mdp.T(s, s_prime, current_action)
                    reward_s_prime = mdp.R(s_prime_idx)  # R expects 0-indexed state
                    # Use Vprev for the value of the next state (from previous iteration)
                    expected_value_for_s += prob * (reward_s_prime + mdp.discountf * Vprev[s_prime_idx])
            V_evaluated_for_current_policy[s_idx] = expected_value_for_s

        Vprev = list(V_evaluated_for_current_policy)  # Update Vprev with the newly evaluated values

        # --- Policy Improvement Step ---
        temp_policy_snapshot = list(newPolicy)  # Take a snapshot of the policy before potential changes

        for s_idx in range(9):  # Iterate through states (0-indexed)
            s = s_idx + 1  # Convert to 1-indexed for MDP functions

            # The original code implied that if newPolicy[s_idx] is 0, its policy is not improved.
            # This suggests 0 might represent a terminal state or a fixed policy.
            # If this is not the desired behavior, this `if` condition should be removed.
            if temp_policy_snapshot[s_idx] != 0: # Only improve policy if it's not a 'fixed' 0 policy
                action_values = np.zeros(len(mdp.action))  # Q-values for each possible action

                for action_idx, action in enumerate(mdp.action): # Iterate through all available actions
                    q_value_for_action = 0.0
                    # Sum over all possible next states s_prime
                    for s_prime_idx in range(9):
                        s_prime = s_prime_idx + 1
                        if (s, s_prime, action) in mdp.transition:
                            prob = mdp.T(s, s_prime, action)
                            reward_s_prime = mdp.R(s_prime_idx)
                            # Use the latest evaluated value function (Vprev)
                            q_value_for_action += prob * (reward_s_prime + mdp.discountf * Vprev[s_prime_idx])
                    action_values[action_idx] = q_value_for_action

                # Find the action with the maximum Q-value for the current state
                best_action_for_state = mdp.action[np.argmax(action_values)]

                # Update the policy if a better action is found
                if newPolicy[s_idx] != best_action_for_state:
                    policy_changed_count += 1
                    newPolicy[s_idx] = best_action_for_state

        # Check for policy convergence
        if policy_changed_count == 0:
            print(f"Early Stop at iteration {itr} due to no policy change.")
            break

    return newPolicy

In [11]:
import numpy as np
actions = [1,-1,2,-2]
r_values = [100, 3, 0, -3]
policy = np.array([0,2,0,1,1,1,1,1,1])
newpolicy = np.array([[0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0]])
index = 0
for r in r_values:
    states = [r, -1, 10, -1, -1, -1, -1, -1, -1]
    start = 8
    mdp = MDP(states, actions, transitions, start, policy, discountf=0.99)
    newpolicy[index] = PolicyIteration(8, mdp)
    print('policy for r = ', r)
    print(newpolicy[index])
    print("path from start point")
    getpath(newpolicy[index],start)
    index = index + 1
print("all", newpolicy)

Early Stop at iteration 3 due to no policy change.
policy for r =  100
[ 0 -2  0  1 -2 -2  1 -2  1]
path from start point
Left
Up
Up
Early Stop at iteration 0 due to no policy change.
policy for r =  3
[0 2 0 1 1 1 1 1 1]
path from start point
Up
Up
Right
Early Stop at iteration 3 due to no policy change.
policy for r =  0
[0 2 0 2 1 1 1 2 1]
path from start point
Right
Up
Up
Early Stop at iteration 3 due to no policy change.
policy for r =  -3
[0 2 0 2 1 1 2 2 1]
path from start point
Right
Up
Up
all [[ 0 -2  0  1 -2 -2  1 -2  1]
 [ 0  2  0  1  1  1  1  1  1]
 [ 0  2  0  2  1  1  1  2  1]
 [ 0  2  0  2  1  1  2  2  1]]


In [12]:
actions = [1,-1,2,-2]
r_values = [100, 3, 0, -3]
policy = [0,2,0,1,1,1,1,1,1] # changed for exit states to 0
#states = [r, -1, 10, -1, -1, -1, -1, -1, -1]